## Proceso de Ornstein-Uhlenbeck

El proceso de Ornstein-Uhlenbeck es un modelo estocástico clásico para describir sistemas con tendencia a regresar a un valor promedio (proceso de reversión a la media). Es ampliamente utilizado en física, biología y finanzas para modelar ruido coloreado, movimiento de partículas en potenciales armónicos y tasas de interés, entre otros.

La ecuación diferencial estocástica (EDS) que lo describe es:

$$
dx = -\gamma x\,\Delta t + \sqrt{2D}\,dW_t
$$

donde:
- $x$ es la variable del proceso,
- $\gamma$ es la tasa de reversión a la media,
- $D$ es el coeficiente de difusión,
- $dW_t$ es el incremento de un proceso de Wiener (ruido blanco gaussiano).

El proceso tiende a regresar a $x=0$ con una fuerza proporcional a $-\gamma x$, mientras que el término estocástico introduce fluctuaciones aleatorias.

La media y la varianza del proceso son:
- $\langle x(t) \rangle = x_0 e^{-\gamma t}$
- $\langle x^2(t) \rangle = \frac{D}{\gamma} \left(1 - e^{-2\gamma t}\right)$

En el límite largo ($t \to \infty$), la varianza se estabiliza en $D/\gamma$.

Este proceso es fundamental para entender la difusión con memoria y la dinámica de sistemas con tendencia a la estabilidad.

In [ ]:
using Plots, Random, Distributions, Statistics #Paqueterías a usar

In [ ]:
function Ornstein_Uhlenbeck_process(steps, x0, Δt, γ, D)
    n = steps
    ξ = randn(n)                      
    x = zeros(n+1)
    x[1] = x0
    for i in 1:n
        x[i+1] = x[i] - γ*x[i]*Δt + sqrt(2*D*Δt)*ξ[i]
    end
    return  x
end

In [ ]:
function generate_trajectories(γ, D, Δt, steps, x0, number_of_trajectories)
    trajectories = []
    for _ in 1:number_of_trajectories
        push!(trajectories, Ornstein_Uhlenbeck_process(steps, x0, Δt, γ, D))
    end
    return trajectories
end

In [ ]:
function plot_trajectories(steps, Δt, trajectories) # Función para generar gráficas de múltiples trayectorias
    p = plot() #Nueva gráfica
    for traj in trajectories
        plot!(p, 0:Δt:Δt*steps, traj, label="", alpha=0.5)
    end
    xlabel!(p, "Time")
    ylabel!(p, "Position")
    title!(p, "Ornstein-Uhlenbeck Trajectories")
    return p # Mostrar gráfica generadas
end

In [ ]:
function compute_msd(trajectories) #Función para calcular el MSD a partir de múltiples trayectorias
    msd_sum = zeros(steps + 1)
    for i in 1:number_of_trajectories
        traj = trajectories[i]
        msd_sum .+= traj.^2
    end
    msd = msd_sum / number_of_trajectories
    return msd
end

In [ ]:
steps, γ, D, Δt, number_of_trajectories, x0 = 10000, 1.0, 1.0, 0.1, 100, 0.0
trajectories = generate_trajectories(γ, D, Δt, steps, x0, number_of_trajectories)
plot_trajectories(steps, Δt, trajectories)

In [ ]:
function theoretical_msd(t,D, γ)
    return (D/γ) .* (1 .- exp.(-2*γ*t) ) 
end

In [ ]:
function plot_msd(msd, steps, Δt, D, γ)
    plot(msd, lw=2, label="Computed MSD")
    plot!(theoretical_msd.((0:steps-1)*Δt, D, γ), lw=2, ls=:dash, label="Theoretical MSD", color=:red)

    xlabel!("Time Steps")
    ylabel!("Mean Squared Displacement")
    title!("Mean Squared Displacement of Ornstein-Uhlenbeck Process")
    display(current())
end

In [ ]:
plot_msd(compute_msd(trajectories), steps, Δt, D, γ)